In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from datasets import load_dataset
dataset_name = "ridwanFatur98/synth-invoice"
train_dataset = load_dataset(dataset_name, split="train")
sample = train_dataset[0]

In [9]:
from scripts.image_process import process_image, image_to_tensor

# Input Config
input_size = [1280, 960]
align_long_axis = False
random_padding = True

# Input Encoder
img = process_image(sample["image"], input_size, align_long_axis, random_padding)
image_tensors = image_to_tensor(img)
image_tensors = image_tensors.unsqueeze(0)
image_tensors = image_tensors.to(device)

In [4]:
from scripts.tokenizer import get_tokenizer
tokenizer = get_tokenizer()

In [5]:
from scripts.ground_truth_process import process_ground_truth, get_train_inputs

task_start_token = '<s_cord-v2>'
prompt_end_token= '<s_cord-v2>'

max_length = 768
ignore_id: int = -100
prompt_end_token_id = tokenizer.convert_tokens_to_ids(prompt_end_token)

ground_truth = sample["ground_truth"]
gt_token_sequence, _ = process_ground_truth(
    tokenizer,
    ground_truth,
    task_start_token=task_start_token,
)
processed_parse = gt_token_sequence 

input_ids, labels = get_train_inputs(tokenizer, processed_parse, prompt_end_token_id, max_length, ignore_id)
input_ids = input_ids.unsqueeze(0).to(device)
labels = labels.unsqueeze(0).to(device)

### Full Donut Model

In [8]:
import torch.nn as nn
import torch
from scripts.model import DonutConfig, DonutModel

In [10]:
# Encoder Config
encoder_layer = [1, 1, 1, 1]
window_size = 10
input_size = [1280, 960]

# Decoder Config
max_position_embeddings = 768
decoder_layer = 1

In [11]:
config = DonutConfig(
    input_size=(1280, 960),
    window_size=10,
    encoder_layer=(1, 1, 1, 1),
    max_position_embeddings=768,
    decoder_layer=1,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
)

In [12]:
donut_model = DonutModel(config)

In [13]:
donut_model.to(device)
pass

In [14]:
sum(p.numel() for p in donut_model.parameters())

96063388

In [15]:
decoder_input_ids = input_ids[:, :-1]
decoder_labels = labels[:, 1:]

decoder_input_ids.shape, decoder_labels.shape

(torch.Size([1, 767]), torch.Size([1, 767]))

In [16]:
decoder_outputs = donut_model(
    image_tensors=image_tensors,
    decoder_input_ids=decoder_input_ids,
    decoder_labels=decoder_labels,
)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


In [17]:
decoder_outputs['loss']

tensor(16.3145, device='cuda:0', grad_fn=<NllLossBackward0>)